# Phase 5 — Feature Engineering Expansion

**Project:** “Machine Learning approaches for accelerating antibiotic susceptibility testing (AST) in microfluidic systems” 

**Date:** May 2026 

## 1. Introduction

This notebook presents the additional feature engineering process applied to GVR measurements, deriving time-normalised metrics, cross-tube statistical summaries, and growth-related indicators to capture both intra-sample variability and overall experimental dynamics.

### Objectives
- Generate additional features from the original GVR measurements
- Capture temporal growth dynamics through time-normalised metrics
- Extract experiment-level and tube-level summary statistics
- Provide a richer representation of bacterial growth behaviour

### New Features
| Feature | Description |
|---|---|
| `auc_per_time` | AUC normalised by time |
| `delta_per_time` | Delta normalised by time |
| `derivative_per_time` | Derivative normalised by time |
| `mean_gvr` | Mean GVR across all tubes at timepoint t |
| `max_gvr_tube` | Cumulative max GVR of the tube up to t |
| `min_gvr_tube` | Cumulative min GVR of the tube up to t |
| `max_gvr_experiment` | Max GVR across all tubes at timepoint t |
| `min_gvr_experiment` | Min GVR across all tubes at timepoint t |
| `growth_score` | Percentage change in GVR from t-1 to t |
| `growth_score_experiment` | Percentage change in mean GVR across tubes |

## Table of Contents
1. [Introduction](#1-introduction)
2. [Imports & Setup](#2-imports--setup)
3. [Data Loading](#3-data-loading)
4. [Dataset Parsing](#4-dataset-parsing)
5. [Feature Engineering](#5-feature-engineering)
6. [Label Assignment](#6-label-assignment)
7. [Quality Check](#7-quality-check)
8. [Save Dataset](#8-save-dataset)

## 2. Imports & Setup

In [1]:
# Data manipulation
import numpy as np
import pandas as pd

# Serialisation
import pickle

# Project utilities
from Preprocessing_functions_1 import *

## 3. Data Loading

### 3.1 Load feature matrix

In [2]:
# Load the base feature matrix from Phase 2 (without label)
for_ml = pd.read_csv("ecoli_ampli_ml.csv")
for_ml = for_ml.drop(columns=["label"])

print(f"Dataset shape: {for_ml.shape}")
for_ml.head()

Dataset shape: (8420, 13)


,experiencia,tubo_id,tempo,antibiotico,concentração,gvr,std_inicial,std_atual,std_tubo,slope,delta,AUC,Derivada
0,20230508_5e6,20230508_5e6_Positive control,0.000000,Ampicillin,Positive control,0.069883,0.008784,0.008784,0.000000,0.000000,0.000000,0.000000,0.000000
1,20230508_5e6,20230508_5e6_Positive control,0.166667,Ampicillin,Positive control,0.069883,0.008784,0.008784,0.000000,0.000000,0.000000,0.011647,0.000000
2,20230508_5e6,20230508_5e6_Positive control,0.333333,Ampicillin,Positive control,-0.139765,0.008784,0.017568,0.098829,-0.628944,-0.209648,0.005824,-1.257887
3,20230508_5e6,20230508_5e6_Positive control,0.500000,Ampicillin,Positive control,-0.199132,0.008784,0.025127,0.121492,-0.538028,-0.269014,-0.022418,-0.356198
4,20230508_5e6,20230508_5e6_Positive control,0.666667,Ampicillin,Positive control,-0.119479,0.008784,0.024350,0.112185,-0.284042,-0.189362,-0.048969,0.477916


### 3.2 Load raw dataset

In [3]:
with open("ecoli_ampli.pkl", "rb") as f:
    ecoli_ampli = pickle.load(f)

print("Raw dataset loaded successfully.")

Raw dataset loaded successfully.


## 4. Dataset Parsing

In [4]:
# Extract experiments, tube IDs, time series and GVR values
antibiotico = "Ampicillin"
data_exp = []
dados = {}
tubo = []
tempo = {}


for data, concs in ecoli_ampli.items():
    for conc, datasets in concs.items():
        df = datasets["gvr"]
        chave = data + "_" + conc
        data_exp.append(chave)

        colunas = df.columns[1:]
        dados[chave] = {}
        tempo[chave] = {}

        for col in colunas:
            t = col[:-2]
            if t not in tubo:
                tubo.append(t)
            else:
                None
            dados[chave][t] = df[col].values
            tempo[chave][t] = df["Time(hrs)"].values

## 5. Feature Engineering

New features are computed at each timepoint, capturing both tube-level and experiment-level growth dynamics.

### 5.1 Growth score helper function

In [5]:
def get_growth_score(anterior, atual):
    """Percentage change in GVR from the previous to the current timepoint."""
    return ((atual - anterior) / anterior) * 100

In [6]:
gvr_media = []        # por experiencia 
gvr_max_tubo = []     # por tubo
gvr_min_tubo = []     # por tubo
gvr_max_experiencia = []  # por experiencia
gvr_min_experiencia = []  # por experiencia
growth_score = []
growth_score_exp = []

for experiencia in data_exp:
    data, conc = experiencia.split("_")
    df = ecoli_ampli[data][conc]["gvr"]
    
    for concentração in tubo:
        gvr_serie = dados[experiencia][concentração]
        tempo_serie = tempo[experiencia][concentração]
        
        for t in range(len(tempo_serie)):
            gvr_media.append(df.iloc[t, 1:].mean())        # média de todos os tubos naquele t
            gvr_max_tubo.append(max(gvr_serie[:t+1]))       # máximo acumulado do tubo até t
            gvr_min_tubo.append(min(gvr_serie[:t+1]))       # mínimo acumulado do tubo até t
            gvr_max_experiencia.append(df.iloc[t, 1:].max()) # máximo entre tubos naquele t
            gvr_min_experiencia.append(df.iloc[t, 1:].min()) # mínimo entre tubos naquele t
            if t == 0:
                growth_score.append(np.nan)
                growth_score_exp.append(np.nan)
            else:
                growth_score.append(get_growth_score(gvr_serie[t-1], gvr_serie[t]))
                growth_score_exp.append(get_growth_score(df.iloc[t-1, 1:].mean(), df.iloc[t, 1:].mean()))


C:\Users\eduar\AppData\Local\Temp\ipykernel_8960\1061659367.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  return ((atual - anterior) / anterior) * 100


### 5.3 Add normalised and new features to DataFrame

In [7]:
for_ml['auc_por_tempo'] = for_ml['AUC'] / for_ml['tempo']
for_ml['delta_por_tempo'] = for_ml['delta'] / for_ml['tempo']
for_ml['Derivada_por_tempo'] = for_ml['Derivada'] / for_ml['tempo']
for_ml["gvr_media"] = gvr_media
for_ml["gvr_max_tubo"] = gvr_max_tubo
for_ml["gvr_min_tubo"] = gvr_min_tubo
for_ml["gvr_max_experiencia"] = gvr_max_experiencia
for_ml["gvr_min_experiencia"] = gvr_min_experiencia
for_ml["growth_score"] = growth_score
for_ml["growth_score_exp"] = growth_score_exp

# Fill NaN at t=0 (no previous timepoint) with 0

nan_cols = ["auc_por_tempo", "delta_por_tempo", "Derivada_por_tempo", "growth_score", "growth_score_exp"]
for_ml[nan_cols] = for_ml[nan_cols].fillna(0)

## 6. Label Assignment

In [8]:
# Assign binary growth label based on final AUC per tube
for id_tubo in for_ml["tubo_id"].unique():
    subset = for_ml[for_ml["tubo_id"] == id_tubo]
    ultimo = subset.iloc[-1]
    
    label = "1" if ultimo["AUC"] > 0 else "0"
    
    for_ml.loc[for_ml["tubo_id"] == id_tubo, "label"] = label


In [9]:
display(for_ml)

,experiencia,tubo_id,tempo,antibiotico,concentração,gvr,std_inicial,std_atual,std_tubo,slope,...,delta_por_tempo,Derivada_por_tempo,gvr_media,gvr_max_tubo,gvr_min_tubo,gvr_max_experiencia,gvr_min_experiencia,growth_score,growth_score_exp,label
0,20230508_5e6,20230508_5e6_Positive control,0.000000,Ampicillin,Positive control,0.069883,0.008784,0.008784,0.000000,0.000000,...,0.000000,0.000000,0.066011,0.069883,0.069883,0.077113,0.048811,0.000000,0.000000,1
1,20230508_5e6,20230508_5e6_Positive control,0.166667,Ampicillin,Positive control,0.069883,0.008784,0.008784,0.000000,0.000000,...,0.000000,0.000000,0.066011,0.069883,0.069883,0.077113,0.048811,0.000000,0.000000,1
2,20230508_5e6,20230508_5e6_Positive control,0.333333,Ampicillin,Positive control,-0.139765,0.008784,0.017568,0.098829,-0.628944,...,-0.628944,-3.773662,-0.132023,0.069883,-0.139765,-0.097623,-0.154226,-300.000000,-300.000000,1
3,20230508_5e6,20230508_5e6_Positive control,0.500000,Ampicillin,Positive control,-0.199132,0.008784,0.025127,0.121492,-0.538028,...,-0.538028,-0.712396,-0.173866,0.069883,-0.199132,-0.133680,-0.206227,42.475728,31.694078,1
4,20230508_5e6,20230508_5e6_Positive control,0.666667,Ampicillin,Positive control,-0.119479,0.008784,0.024350,0.112185,-0.284042,...,-0.284042,0.716874,-0.088582,0.069883,-0.199132,-0.048963,-0.127436,-40.000000,-49.051471,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8415,20230518_5e4,20230518_5e4_128,15.666667,Ampicillin,128,-0.203146,0.006013,1.223849,0.074035,-0.014303,...,-0.014303,0.010413,0.928440,0.021168,-0.245417,2.579524,-0.203146,-11.803884,4.318240,0
8416,20230518_5e4,20230518_5e4_128,15.833333,Ampicillin,128,-0.230334,0.006013,1.225105,0.074306,-0.015870,...,-0.015870,-0.010303,0.890007,0.021168,-0.245417,2.530699,-0.230334,13.383678,-4.139487,0
8417,20230518_5e4,20230518_5e4_128,16.000000,Ampicillin,128,-0.216759,0.006013,1.211603,0.074390,-0.014856,...,-0.014856,0.005091,0.906519,0.021168,-0.245417,2.543638,-0.216759,-5.893547,1.855177,0
8418,20230518_5e4,20230518_5e4_128,16.166667,Ampicillin,128,-0.197151,0.006013,1.210670,0.074272,-0.013490,...,-0.013490,0.007277,0.921314,0.021168,-0.245417,2.561307,-0.197151,-9.046033,1.632128,0


## 7. Quality Check

In [10]:
# Check for missing values
print("Missing values per column:")
print(for_ml.isnull().sum())

# Inspect first row
print(f"\nDataset shape: {for_ml.shape}")
display(for_ml.iloc[0])

Missing values per column:
experiencia            0
tubo_id                0
tempo                  0
antibiotico            0
concentração           0
gvr                    0
std_inicial            0
std_atual              0
std_tubo               0
slope                  0
delta                  0
AUC                    0
Derivada               0
auc_por_tempo          0
delta_por_tempo        0
Derivada_por_tempo     0
gvr_media              0
gvr_max_tubo           0
gvr_min_tubo           0
gvr_max_experiencia    0
gvr_min_experiencia    0
growth_score           0
growth_score_exp       0
label                  0
dtype: int64

Dataset shape: (8420, 24)


experiencia                             20230508_5e6
tubo_id                20230508_5e6_Positive control
tempo                                            0.0
antibiotico                               Ampicillin
concentração                        Positive control
gvr                                         0.069883
std_inicial                                 0.008784
std_atual                                   0.008784
std_tubo                                         0.0
slope                                            0.0
delta                                            0.0
AUC                                              0.0
Derivada                                         0.0
auc_por_tempo                                    0.0
delta_por_tempo                                  0.0
Derivada_por_tempo                               0.0
gvr_media                                   0.066011
gvr_max_tubo                                0.069883
gvr_min_tubo                                0.

## 8. Save Dataset

In [11]:
# Save expanded feature matrix for use in Phase 6
for_ml.to_csv("ecoli_ampli_expanded.csv", index=False)

print("Dataset saved successfully.")
print(f"  Shape  : {for_ml.shape}")
print(f"  Labels : {for_ml['label'].value_counts()}")

Dataset saved successfully.
  Shape  : (8420, 24)
  Labels : label
1    4949
0    3471
Name: count, dtype: int64
